# Fine-tuning Sentimen Bahasa Indonesia — GPU Gratis (Colab)

Notebook ini melatih model sentimen memakai **GPU gratis Google Colab**, karena
melatih di CPU laptop sangat lambat.

**Cara pakai:**
1. Buka di [Google Colab](https://colab.research.google.com/) → `File > Upload notebook`
2. Aktifkan GPU: `Runtime > Change runtime type > Hardware accelerator: GPU`
3. Jalankan sel dari atas ke bawah
4. Unduh model hasilnya, taruh di `models/` pada proyekmu

> **Penting:** gunakan data berlabel **manusia**. Melatih dari label keluaran
> model itu sendiri bersifat sirkular — model hanya meniru dirinya.

In [ ]:
# 1) Pasang dependensi
!pip -q install transformers datasets scikit-learn accelerate

In [ ]:
# 2) Pastikan GPU aktif
import torch
print('GPU tersedia :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Perangkat    :', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: GPU belum aktif. Runtime > Change runtime type > GPU')

## 3) Ambil data latih

Pilih **salah satu**: dataset berlabel dari HuggingFace, atau CSV milikmu
(`text,label`) yang di-upload ke Colab.

In [ ]:
# OPSI A — dataset berlabel manusia dari HuggingFace
from datasets import load_dataset
from itertools import islice
from collections import Counter

JUMLAH = 20000   # perbesar bila ingin hasil lebih baik

ds = load_dataset('carant-ai/indonesian_sentiment_dataset', split='train', streaming=True)
teks, label = [], []
for row in islice(ds, JUMLAH):
    t = (row.get('text') or '').strip()
    l = str(row.get('label_text') or '').lower()
    if t and l in ('positive', 'neutral', 'negative'):
        teks.append(t); label.append(l)

# seimbangkan antar kelas
per = {}
for t, l in zip(teks, label):
    per.setdefault(l, []).append(t)
n = min(len(v) for v in per.values())
teks, label = [], []
for l, items in per.items():
    teks += items[:n]; label += [l] * n

print(len(teks), 'contoh —', Counter(label))

In [ ]:
# OPSI B — CSV milikmu sendiri (kolom: text,label)
# from google.colab import files
# import csv, io
# up = files.upload()
# nama = list(up)[0]
# teks, label = [], []
# for row in csv.DictReader(io.StringIO(up[nama].decode('utf-8'))):
#     teks.append(row['text']); label.append(row['label'].lower())
# print(len(teks), 'contoh')

In [ ]:
# 4) Latih
import numpy as np, torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

MODEL_DASAR = 'indobenchmark/indobert-base-p1'   # atau 'w11wo/indonesian-roberta-base-sentiment-classifier'
EPOCHS, BATCH, LR, MAXLEN = 3, 32, 2e-5, 128

kelas = sorted(set(label))
l2i = {l: i for i, l in enumerate(kelas)}
i2l = {i: l for l, i in l2i.items()}
y = [l2i[l] for l in label]

Xtr, Xte, ytr, yte = train_test_split(teks, y, test_size=0.2, random_state=42, stratify=y)
tok = AutoTokenizer.from_pretrained(MODEL_DASAR)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DASAR, num_labels=len(kelas), id2label=i2l, label2id=l2i,
    ignore_mismatched_sizes=True).cuda()

def ds_of(X, Y):
    e = tok(X, truncation=True, padding='max_length', max_length=MAXLEN, return_tensors='pt')
    return TensorDataset(e['input_ids'], e['attention_mask'], torch.tensor(Y))

dl_tr = DataLoader(ds_of(Xtr, ytr), batch_size=BATCH, shuffle=True)
dl_te = DataLoader(ds_of(Xte, yte), batch_size=BATCH)
opt = torch.optim.AdamW(model.parameters(), lr=LR)

for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0
    for ids, mask, yy in dl_tr:
        ids, mask, yy = ids.cuda(), mask.cuda(), yy.cuda()
        opt.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=yy)
        out.loss.backward(); opt.step(); tot += out.loss.item()
    model.eval(); pred = []
    with torch.no_grad():
        for ids, mask, _ in dl_te:
            pred += model(input_ids=ids.cuda(), attention_mask=mask.cuda()).logits.argmax(-1).cpu().tolist()
    acc = np.mean(np.array(pred) == np.array(yte))
    print(f'epoch {ep}: loss {tot/len(dl_tr):.4f} | akurasi {acc:.4f}')

print()
print(classification_report(yte, pred, target_names=kelas))

In [ ]:
# 5) Simpan & unduh
OUT = 'indobert-sentimen-finetuned'
model.save_pretrained(OUT); tok.save_pretrained(OUT)
!zip -qr {OUT}.zip {OUT}
from google.colab import files
files.download(f'{OUT}.zip')
print('Ekstrak ke: models/indobert-sentiment-finetuned/ pada proyekmu,')
print('lalu set config.yaml -> sentiment.model_dir: "models/indobert-sentiment-finetuned"')

In [ ]:
# 6) (Opsional) unggah ke HuggingFace Hub — REPO PRIVAT
# Token: https://huggingface.co/settings/tokens (akses Write)
# from huggingface_hub import HfApi
# TOKEN = ''            # jangan dibagikan ke siapa pun
# REPO  = 'namamu/indobert-sentimen-ikn'
# api = HfApi(token=TOKEN)
# api.create_repo(REPO, private=True, exist_ok=True)   # privat!
# api.upload_folder(folder_path=OUT, repo_id=REPO)
# print('https://huggingface.co/' + REPO)